# Random Sample of 300 Predictions for Manual Checking

This notebook randomly selects **300 images** from the prediction CSV and displays them in a **5-column gallery** with the predicted `gender`, `articleType`, `season`, and `usage` labels. Use the page slider to inspect the sample without loading all 300 images into memory at once.

In [ ]:
import base64
import io
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ============================================================
# SETTINGS
# ============================================================

CSV_PATH = Path("../outputs/image_only/styles_prediction_filled.csv")
IMAGE_DIR = Path("../data/raw/FashionDataset/test/images_test")

N_RANDOM = 300
N_COLUMNS = 5
IMAGES_PER_PAGE = 15   # 5 columns x 3 rows
RANDOM_STATE = 42       # Change this for a different random sample
THUMBNAIL_SIZE = (180, 220)

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".webp"]

In [ ]:
# ============================================================
# LOAD PREDICTIONS AND SELECT 300 RANDOM IMAGES
# ============================================================

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Prediction CSV not found: {CSV_PATH}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}")

required = ["id", "gender", "articleType", "season", "usage"]
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing CSV columns: {missing}. Found: {list(df.columns)}")

df = df[required].dropna().copy()

sample_n = min(N_RANDOM, len(df))
sample_df = df.sample(n=sample_n, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Loaded {len(df):,} prediction rows.")
print(f"Random sample selected: {len(sample_df):,} images.")
print(f"Random state: {RANDOM_STATE}")

In [ ]:
# ============================================================
# IMAGE HELPERS
# ============================================================

def find_image(image_id):
    """Find the image file corresponding to an ID."""
    image_id = str(image_id).strip()
    for ext in IMAGE_EXTENSIONS:
        path = IMAGE_DIR / f"{image_id}{ext}"
        if path.exists():
            return path
    return None


def image_to_data_uri(path):
    """Create a small JPEG data URI for embedding in the notebook HTML."""
    with Image.open(path) as img:
        img = img.convert("RGB")
        img.thumbnail(THUMBNAIL_SIZE)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=82, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def render_page(page_number):
    """Render one page of 15 predictions (5 columns x 3 rows)."""
    start = page_number * IMAGES_PER_PAGE
    end = min(start + IMAGES_PER_PAGE, len(sample_df))
    page_df = sample_df.iloc[start:end]

    cards = []

    for _, row in page_df.iterrows():
        image_id = str(row["id"])
        image_path = find_image(image_id)

        if image_path is not None:
            try:
                image_html = (
                    f'<img src="{image_to_data_uri(image_path)}" '
                    'style="width:180px;height:220px;object-fit:contain;display:block;margin:auto;">'
                )
            except Exception as exc:
                image_html = (
                    f'<div style="height:220px;display:flex;align-items:center;'
                    f'justify-content:center;text-align:center;">Could not open image<br>{exc}</div>'
                )
        else:
            image_html = (
                '<div style="height:220px;display:flex;align-items:center;'
                'justify-content:center;text-align:center;">'
                f'Image not found<br>ID: {image_id}</div>'
            )

        card = f'''
        <div style="box-sizing:border-box;width:220px;margin:8px;padding:10px;
                    border:1px solid #d0d7de;border-radius:8px;background:#fff;
                    font-family:Arial,sans-serif;vertical-align:top;">
            {image_html}
            <div style="margin-top:8px;font-size:12px;line-height:1.45;">
                <b>ID:</b> {image_id}<br>
                <b>Gender:</b> {row['gender']}<br>
                <b>Type:</b> {row['articleType']}<br>
                <b>Season:</b> {row['season']}<br>
                <b>Usage:</b> {row['usage']}
            </div>
        </div>
        '''
        cards.append(card)

    gallery = "".join(cards)
    total_pages = max(1, (len(sample_df) + IMAGES_PER_PAGE - 1) // IMAGES_PER_PAGE)

    html = f'''
    <div style="font-family:Arial,sans-serif;margin:5px 0 10px 0;">
        <h3 style="margin:0 0 4px 0;">Random Prediction Sample</h3>
        <div style="font-size:13px;color:#555;">
            Showing <b>{start + 1}-{end}</b> of <b>{len(sample_df)}</b> sampled images
            &nbsp;|&nbsp; Page <b>{page_number + 1}</b> of <b>{total_pages}</b>
        </div>
    </div>
    <div style="display:flex;flex-wrap:wrap;align-items:flex-start;justify-content:center;
                max-height:900px;overflow-y:auto;padding:5px;background:#f6f8fa;
                border:1px solid #d8dee4;border-radius:8px;">
        {gallery}
    </div>
    '''

    return html

In [ ]:
# ============================================================
# INTERACTIVE 5-COLUMN GALLERY — BUTTONS + SWIPE
# ============================================================

total_pages = max(1, (len(sample_df) + IMAGES_PER_PAGE - 1) // IMAGES_PER_PAGE)
current_page = 0

prev_button = widgets.Button(
    description="← Previous",
    disabled=True,
    layout=widgets.Layout(width="120px")
)

next_button = widgets.Button(
    description="Next →",
    disabled=(total_pages <= 1),
    layout=widgets.Layout(width="120px")
)

page_label = widgets.HTML()
output = widgets.Output()

def update_gallery():
    global current_page

    prev_button.disabled = current_page == 0
    next_button.disabled = current_page >= total_pages - 1
    page_label.value = f"<b>Page {current_page + 1} of {total_pages}</b>"

    with output:
        clear_output(wait=True)
        display(HTML(render_page(current_page)))

def go_previous(_=None):
    global current_page
    if current_page > 0:
        current_page -= 1
        update_gallery()

def go_next(_=None):
    global current_page
    if current_page < total_pages - 1:
        current_page += 1
        update_gallery()

prev_button.on_click(go_previous)
next_button.on_click(go_next)

controls = widgets.HBox(
    [prev_button, next_button, page_label],
    layout=widgets.Layout(align_items="center")
)

display(controls)
display(output)

# ------------------------------------------------------------
# Swipe support
# ------------------------------------------------------------
# A small JavaScript listener detects horizontal touch swipes
# anywhere on the notebook page. Left = next, right = previous.
# It sends a click to the corresponding ipywidget button.

swipe_js = HTML("""
<script>
(function() {
    let startX = null;
    let startY = null;
    const threshold = 70;

    document.addEventListener('touchstart', function(e) {
        if (!e.touches || e.touches.length !== 1) return;
        startX = e.touches[0].clientX;
        startY = e.touches[0].clientY;
    }, {passive: true});

    document.addEventListener('touchend', function(e) {
        if (startX === null || !e.changedTouches || e.changedTouches.length !== 1) return;

        const endX = e.changedTouches[0].clientX;
        const endY = e.changedTouches[0].clientY;
        const dx = endX - startX;
        const dy = endY - startY;

        startX = null;
        startY = null;

        // Only treat predominantly horizontal gestures as swipes.
        if (Math.abs(dx) < threshold || Math.abs(dx) <= Math.abs(dy)) return;

        // Find the notebook widget buttons by their visible text.
        const buttons = Array.from(document.querySelectorAll('button'));
        const previous = buttons.find(b => b.innerText.includes('Previous'));
        const next = buttons.find(b => b.innerText.includes('Next'));

        if (dx < 0 && next && !next.disabled) {
            next.click();
        } else if (dx > 0 && previous && !previous.disabled) {
            previous.click();
        }
    }, {passive: true});
})();
</script>
""")
display(swipe_js)

update_gallery()

### Manual checking notes

The sample uses a fixed `RANDOM_STATE = 42`, so the same 300 images are selected each time the notebook is run. To inspect a different random sample, change `RANDOM_STATE` and rerun the sampling cell and gallery cells.

For manual evaluation, compare each displayed prediction against the actual garment image and record any incorrect `gender`, `articleType`, `season`, or `usage` predictions separately if needed.